# 04 — The Precursor Test

**Do topological signatures systematically precede breakthroughs?**

This is the hypothesis test. For each cataloged breakthrough, we compare the
topological metrics (β₁, persistence entropy) in the years *before* the
breakthrough against a null model — the same CPC section pair at times when
no breakthrough occurred.

Key method: **Superposed epoch analysis** — align all breakthroughs at t=0
and average the signal. If topology is a precursor, we should see a systematic
rise before t=0 that the null model doesn't exhibit.

---

In [ ]:
# %% Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from tqdm import tqdm

from src.utils import DATA_DIR, get_logger
from src.breakthroughs import load_breakthroughs, get_precursor_window
from src.topology import compute_all_priority_pairs, ALL_PAIRS
from src.nullmodel import matched_null, random_cpc_pair_baseline, superposed_epoch
from src.plotting import set_style, save_figure, PALETTE, PALETTE_LIST, year_axis, confidence_band

set_style()
logger = get_logger('nb04')

CACHE_DIR = str(DATA_DIR / 'topology_cache')

In [2]:
# %% Load data and breakthroughs
patents = pd.read_parquet(DATA_DIR / 'patents.parquet')
citations = pd.read_parquet(DATA_DIR / 'citations.parquet')
cpc_map = pd.read_parquet(DATA_DIR / 'cpc_map.parquet')

# Ensure citing_year column exists (required by new topology module)
citations['citing_year'] = pd.to_datetime(citations['citing_date']).dt.year
citations.drop(columns=['citing_date'], inplace=True)  # Free ~1GB RAM

breakthroughs = load_breakthroughs()
print(f'Breakthroughs: {len(breakthroughs)}')

2026-03-20 20:19:00 | src.breakthroughs        | INFO    | Loaded 34 breakthroughs from 8 files


Breakthroughs: 34


In [3]:
# %% Load topology results from Notebook 02 (all 28 pairs, scale-normalized)
import gc

pair_results = compute_all_priority_pairs(
    citations, cpc_map,
    cache_dir=CACHE_DIR,
    pairs=ALL_PAIRS,
    start_year=1980,
    end_year=2023,
)

# Build topology_results dict keyed by both "AxC" and "A_C" formats
topology_results = {}
for pair, group in pair_results.groupby('section_pair'):
    topology_results[pair] = group.reset_index(drop=True)
    underscore_key = pair.replace('x', '_')
    topology_results[underscore_key] = group.reset_index(drop=True)

print(f'Total topology results: {len(pair_results["section_pair"].unique())} CPC pairs (of 28)')
print(f'Total window-pair observations: {len(pair_results)}')
gc.collect()

Total topology results: 28 CPC pairs (of 28)
Total window-pair observations: 980


0

## Compute Matched Null Models

For each breakthrough, generate null topology measurements from the same
CPC pair but in non-breakthrough time periods.

In [4]:
# %% Generate matched null models for ALL breakthroughs
# With all 28 CPC pairs cached (scale-normalized), we can compute null models
# for every breakthrough — no more skipping non-priority pairs.
import pickle

null_results = {}
skipped = []
computed_fresh = []

for bt in tqdm(breakthroughs, desc='Computing null models'):
    bt_name = bt.name.replace(" ", "_").replace("/", "_").lower()[:30]
    cache_file = DATA_DIR / "null_cache" / f"matched_{bt_name}_n100_s42.pkl"
    
    if cache_file.exists():
        with open(cache_file, "rb") as f:
            null_results[bt.name] = pickle.load(f)
    else:
        # Compute fresh — should be fast with v2 topology cache
        try:
            null_df = matched_null(
                bt, citations, cpc_map,
                n_samples=100, seed=42, use_cache=True,
            )
            if len(null_df) > 0:
                null_results[bt.name] = null_df
                computed_fresh.append(bt.name)
            else:
                skipped.append(bt.name)
        except Exception as e:
            print(f'  Error for {bt.name}: {e}')
            skipped.append(bt.name)

print(f'\nNull models: {len(null_results)} breakthroughs')
print(f'  Loaded from cache: {len(null_results) - len(computed_fresh)}')
print(f'  Computed fresh: {len(computed_fresh)}')
if skipped:
    print(f'  Skipped: {skipped}')

Computing null models:   0%|          | 0/34 [00:00<?, ?it/s]

Matched null: Carbon Fiber Composites:   0%|          | 0/100 [00:00<?, ?it/s]

Matched null: Carbon Fiber Composites:  35%|███▌      | 35/100 [00:00<00:00, 347.33it/s]

Matched null: Carbon Fiber Composites:  85%|████████▌ | 85/100 [00:00<00:00, 432.44it/s]

Matched null: Carbon Fiber Composites: 100%|██████████| 100/100 [00:00<00:00, 430.62it/s]

...


2026-03-20 20:49:59 | src.nullmodel            | INFO    | Matched null for mRNA Vaccine Platform: 100/100 samples computed


2026-03-20 20:49:59 | timer                    | INFO    | matched_null completed in 0.2s


Computing null models: 100%|██████████| 34/34 [00:07<00:00,  5.18it/s]

Computing null models: 100%|██████████| 34/34 [00:07<00:00,  4.75it/s]


Null models: 34 breakthroughs
  Loaded from cache: 0
  Computed fresh: 34


## Pre-Breakthrough vs. Null Topology

For each breakthrough: extract β₁ in the 10 years before filing,
compare against the matched null distribution.

In [5]:
# %% Extract pre-breakthrough metrics and compute z-scores
# Match breakthroughs to ANY topology pair containing their CPC section(s)
comparison = []

for bt in breakthroughs:
    sections = bt.cpc_sections if len(bt.cpc_sections) >= 1 else []
    if not sections:
        continue
    
    # Find all topology pairs containing at least one of this breakthrough's sections
    matching_topos = []
    for key, topo in topology_results.items():
        if 'x' not in key:
            continue
        sec_a, sec_b = key.split('x')
        if any(s in [sec_a, sec_b] for s in sections):
            matching_topos.append(topo)
    
    if not matching_topos:
        continue
    
    year_col = 'window_end'
    start, end = get_precursor_window(bt, years_before=10)
    
    pre_beta1_vals = []
    pre_pe_vals = []
    for topo in matching_topos:
        if year_col not in topo.columns:
            continue
        pre = topo[(topo[year_col] >= start) & (topo[year_col] <= end)]
        if len(pre) > 0 and 'beta_1' in pre.columns:
            pre_beta1_vals.append(pre['beta_1'].mean())
            pre_pe_vals.append(pre['persistence_entropy'].mean())
    
    if not pre_beta1_vals:
        continue
    
    pre_beta1 = np.mean(pre_beta1_vals)
    pre_pe = np.mean(pre_pe_vals)
    
    null = null_results.get(bt.name)
    if null is None or len(null) == 0:
        continue
    
    null_beta1_mean = null['beta_1'].mean()
    null_beta1_std = null['beta_1'].std()
    null_pe_mean = null['persistence_entropy'].mean()
    null_pe_std = null['persistence_entropy'].std()
    
    # Use percentile rank only when std is truly degenerate (essentially zero)
    DEGEN_THRESH = 1e-6
    
    if null_beta1_std > DEGEN_THRESH:
        z_beta1 = (pre_beta1 - null_beta1_mean) / null_beta1_std
    else:
        pct = (null['beta_1'] < pre_beta1).mean()
        z_beta1 = stats.norm.ppf(max(0.001, min(0.999, pct)))
    
    if null_pe_std > DEGEN_THRESH:
        z_pe = (pre_pe - null_pe_mean) / null_pe_std
    else:
        pct = (null['persistence_entropy'] < pre_pe).mean()
        z_pe = stats.norm.ppf(max(0.001, min(0.999, pct)))
    
    # Count precursor windows available
    n_precursor = 0
    for topo in matching_topos:
        if year_col in topo.columns:
            pre = topo[(topo[year_col] >= start) & (topo[year_col] <= end)]
            n_precursor = max(n_precursor, len(pre))
    
    comparison.append({
        'name': bt.name,
        'category': bt.category,
        'filing_year': bt.filing_year,
        'n_sections': len(sections),
        'n_precursor_windows': n_precursor,
        'pre_beta1': pre_beta1,
        'null_beta1_mean': null_beta1_mean,
        'null_beta1_std': null_beta1_std,
        'z_beta1': z_beta1,
        'pre_pe': pre_pe,
        'null_pe_mean': null_pe_mean,
        'null_pe_std': null_pe_std,
        'z_pe': z_pe,
        'n_matching_pairs': len(matching_topos),
    })

comp_df = pd.DataFrame(comparison)
print(f'Breakthroughs with valid comparisons: {len(comp_df)}')
print(f'\nMean z-score (β₁):  {comp_df["z_beta1"].mean():.3f} ± {comp_df["z_beta1"].std():.3f}')
print(f'Mean z-score (PE):   {comp_df["z_pe"].mean():.3f} ± {comp_df["z_pe"].std():.3f}')
print(f'\nPer-breakthrough results:')
print(comp_df[['name', 'filing_year', 'n_sections', 'n_precursor_windows', 'z_beta1', 'z_pe', 'n_matching_pairs']].to_string(index=False))

Breakthroughs with valid comparisons: 21

Mean z-score (β₁):  -1.772 ± 3.298
Mean z-score (PE):   -26.020 ± 24.637

Per-breakthrough results:
                                           name  filing_year   z_beta1       z_pe  n_matching_pairs
                  Convolutional Neural Networks         1990 -2.617355 -30.830401                 7
  Hydraulic Fracturing with Horizontal Drilling         1990 -3.983011 -51.047970                 7
           Modern Wind Turbine (Variable Speed)         1990 -4.407002 -43.553254                 7
                    Elliptic Curve Cryptography         1991  3.814192   2.601041                13
                              CMOS Image Sensor         1993  3.617991   2.682361                13
                             WiFi (IEEE 802.11)         1993 -4.466650 -52.454356                 7
                       USB Universal Serial Bus         1994  3.390123   2.555507                13
                Recurrent Neural Network (LSTM)         19

## Figure 1: Superposed Epoch Plot (THE KEY RESULT)

Align all breakthroughs at t=0 (filing year), average β₁ from -10 to +5 years.
Compare against the null model's 95% confidence interval.

In [6]:
# %% Figure 1: Superposed epoch analysis
epoch = superposed_epoch(
    breakthroughs=breakthroughs,
    topology_results=topology_results,
    metric='beta_1',
    years_before=10,
    years_after=5,
)

# Generate null epoch (using random baseline)
# Use small sample — most will hit topology cache
null_baseline = random_cpc_pair_baseline(
    citations=citations,
    cpc_map=cpc_map,
    n_samples=50,
    seed=42,
)

fig, ax = plt.subplots(figsize=(12, 7))

if len(epoch) > 0:
    ax.plot(epoch['epoch_year'], epoch['mean'], color=PALETTE['red'], linewidth=2.5,
            label='Pre-breakthrough β₁ (mean)', zorder=5)
    ax.fill_between(
        epoch['epoch_year'],
        epoch['mean'] - epoch['std'],
        epoch['mean'] + epoch['std'],
        alpha=0.2, color=PALETTE['red'], label='±1 SD'
    )

# Null band
if len(null_baseline) > 0:
    null_mean = null_baseline['beta_1'].mean()
    null_std = null_baseline['beta_1'].std()
    ax.axhspan(null_mean - 1.96 * null_std, null_mean + 1.96 * null_std,
               alpha=0.1, color=PALETTE['blue'], label='Null 95% CI')
    ax.axhline(null_mean, color=PALETTE['blue'], linestyle='--', alpha=0.5)

ax.axvline(0, color='gray', linestyle=':', alpha=0.7, label='Breakthrough (t=0)')
ax.set_xlabel('Years Relative to Breakthrough Filing')
ax.set_ylabel('β₁ (Mean Across Breakthroughs)')
ax.set_title('Figure 1: Superposed Epoch Analysis — β₁ Before and After Breakthroughs')
ax.legend()
fig.tight_layout()
save_figure(fig, '04_superposed_epoch')

Random baseline:   0%|          | 0/50 [00:00<?, ?it/s]

Random baseline:  64%|██████▍   | 32/50 [00:00<00:00, 314.06it/s]

Random baseline: 100%|██████████| 50/50 [00:00<00:00, 326.03it/s]


2026-03-20 20:50:00 | src.nullmodel            | INFO    | Random baseline: 50/50 samples computed


2026-03-20 20:50:00 | timer                    | INFO    | random_cpc_pair_baseline completed in 0.2s


2026-03-20 20:50:01 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_superposed_epoch.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_superposed_epoch.png')

## Figure 2: Individual Breakthrough β₁ Curves

In [7]:
# %% Figure 2: Individual breakthrough curves
# Show a diverse selection including previously-skipped breakthroughs
selected_names = [
    'Lithium-Ion Battery', 'PageRank Algorithm',
    'CRISPR-Cas9 Gene Editing', 'mRNA Vaccine Platform',
    'Selective Laser Sintering (SLS) 3D Printing',
    'GPU General-Purpose Computing (CUDA)', 'WiFi (IEEE 802.11)',
    'CAR-T Cell Therapy', 'Convolutional Neural Networks (CNN)',
    'MapReduce Distributed Computing', 'iPhone Multi-Touch Interface',
    'Graphene Synthesis',
]

selected_bts = [bt for bt in breakthroughs if bt.name in selected_names]
# Fill with other breakthroughs if some aren't found
if len(selected_bts) < 9:
    remaining = [bt for bt in breakthroughs if bt.name not in selected_names]
    selected_bts.extend(remaining[:9 - len(selected_bts)])

n_selected = len(selected_bts)
ncols = 3
nrows = (n_selected + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows), sharex=True)
axes = axes.flatten() if n_selected > 1 else [axes]

for i, bt in enumerate(selected_bts):
    if i >= len(axes):
        break
    ax = axes[i]
    sections = bt.cpc_sections if len(bt.cpc_sections) >= 1 else []
    
    # Find all matching topology pairs
    plotted = False
    for key, topo in topology_results.items():
        if 'x' not in key:
            continue
        sec_a, sec_b = key.split('x')
        if any(s in [sec_a, sec_b] for s in sections):
            year_col = 'window_end' if 'window_end' in topo.columns else 'year'
            ax.plot(topo[year_col], topo['beta_1'], linewidth=1.5, alpha=0.6, label=key)
            plotted = True
    
    if plotted:
        ax.axvline(bt.filing_year, color=PALETTE['red'], linestyle='--', alpha=0.7, label='Filing year')
        ax.axvspan(bt.filing_year - 10, bt.filing_year, alpha=0.05, color=PALETTE['orange'])
    
    ax.set_title(bt.name, fontsize=10)
    ax.set_ylabel('β₁')
    if i == 0:
        ax.legend(fontsize=6, loc='upper right')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Figure 2: β₁ Time Series for Individual Breakthroughs\n(colored by CPC section pair, scale-normalized distances)', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '04_individual_beta1')

2026-03-20 20:50:03 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_individual_beta1.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_individual_beta1.png')

## Figure 3: Effect Size Distribution

In [8]:
# %% Figure 3: Z-score distribution
if len(comp_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # β₁ z-scores
    ax = axes[0]
    ax.barh(range(len(comp_df)), comp_df['z_beta1'].values,
            color=[PALETTE['red'] if z > 0 else PALETTE['blue'] for z in comp_df['z_beta1']])
    ax.set_yticks(range(len(comp_df)))
    ax.set_yticklabels(comp_df['name'].values, fontsize=7)
    ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(1.96, color='gray', linestyle='--', alpha=0.3, label='p=0.05')
    ax.axvline(-1.96, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Z-score (β₁)')
    ax.set_title('β₁ Effect Size')
    ax.legend()
    ax.invert_yaxis()

    # PE z-scores
    ax = axes[1]
    ax.barh(range(len(comp_df)), comp_df['z_pe'].values,
            color=[PALETTE['red'] if z > 0 else PALETTE['blue'] for z in comp_df['z_pe']])
    ax.set_yticks(range(len(comp_df)))
    ax.set_yticklabels(comp_df['name'].values, fontsize=7)
    ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(1.96, color='gray', linestyle='--', alpha=0.3)
    ax.axvline(-1.96, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Z-score (Persistence Entropy)')
    ax.set_title('Persistence Entropy Effect Size')
    ax.invert_yaxis()

    fig.suptitle('Figure 3: Pre-Breakthrough Topology vs. Null Model', fontsize=14, y=1.02)
    fig.tight_layout()
    save_figure(fig, '04_effect_sizes')

2026-03-20 20:50:04 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_effect_sizes.png


## Statistical Tests

In [9]:
# %% Aggregate statistical tests
if len(comp_df) > 0:
    # One-sample t-test: are z-scores significantly different from 0?
    t_stat_b1, p_val_b1 = stats.ttest_1samp(comp_df['z_beta1'], 0)
    t_stat_pe, p_val_pe = stats.ttest_1samp(comp_df['z_pe'], 0)
    
    print('=== Aggregate Statistical Tests ===')
    print(f'N breakthroughs with valid comparisons: {len(comp_df)}')
    print(f'\nβ₁ z-scores: mean={comp_df["z_beta1"].mean():.3f}, t={t_stat_b1:.3f}, p={p_val_b1:.4f}')
    print(f'PE z-scores: mean={comp_df["z_pe"].mean():.3f}, t={t_stat_pe:.3f}, p={p_val_pe:.4f}')
    
    # Wilcoxon signed-rank test (non-parametric)
    try:
        w_stat_b1, wp_b1 = stats.wilcoxon(comp_df['z_beta1'])
        w_stat_pe, wp_pe = stats.wilcoxon(comp_df['z_pe'])
        print(f'\nWilcoxon β₁: W={w_stat_b1:.1f}, p={wp_b1:.4f}')
        print(f'Wilcoxon PE: W={w_stat_pe:.1f}, p={wp_pe:.4f}')
    except ValueError as e:
        print(f'Wilcoxon test error: {e}')
        wp_b1 = wp_pe = 1.0
    
    # Holm-Bonferroni correction for multiple comparisons (4 tests)
    raw_pvals = sorted([
        ('β₁ t-test', p_val_b1),
        ('PE t-test', p_val_pe),
        ('β₁ Wilcoxon', wp_b1),
        ('PE Wilcoxon', wp_pe),
    ], key=lambda x: x[1])
    
    print(f'\n=== Holm-Bonferroni Correction (4 tests) ===')
    any_significant = False
    for i, (name, p) in enumerate(raw_pvals):
        adjusted_alpha = 0.05 / (4 - i)
        sig = '**SIGNIFICANT**' if p < adjusted_alpha else 'not significant'
        print(f'  {name}: p={p:.4f}, threshold={adjusted_alpha:.4f} → {sig}')
        if p < adjusted_alpha:
            any_significant = True
    
    # Cohen's d on raw metric values (not inflated z-scores)
    d_b1_raw = comp_df['pre_beta1'].mean() - comp_df['null_beta1_mean'].mean()
    pooled_std_b1 = np.sqrt((comp_df['pre_beta1'].std()**2 + comp_df['null_beta1_std'].mean()**2) / 2)
    d_b1 = d_b1_raw / pooled_std_b1 if pooled_std_b1 > 0 else 0
    
    d_pe_raw = comp_df['pre_pe'].mean() - comp_df['null_pe_mean'].mean()
    pooled_std_pe = np.sqrt((comp_df['pre_pe'].std()**2 + comp_df['null_pe_std'].mean()**2) / 2)
    d_pe = d_pe_raw / pooled_std_pe if pooled_std_pe > 0 else 0
    
    print(f'\nCohen\'s d (β₁, raw): {d_b1:.3f}')
    print(f'Cohen\'s d (PE, raw):  {d_pe:.3f}')
    
    # Caveats
    print(f'\n=== Caveats ===')
    print(f'- Observations are non-independent: breakthroughs share topology pairs')
    print(f'- Mean matching pairs per breakthrough: {comp_df["n_matching_pairs"].mean():.1f}')
    print(f'- Effective sample size is smaller than N={len(comp_df)}')
    print(f'- 8/34 breakthroughs skipped (non-priority CPC pairs)')
    
    # Interpretation
    if any_significant:
        print(f'\n** At least one test survives Holm-Bonferroni correction **')
    else:
        print(f'\n** No tests survive Holm-Bonferroni correction **')
        print('   Topological signatures do not systematically precede breakthroughs.')
        print('   This is a null result — itself a meaningful scientific finding.')

=== Aggregate Statistical Tests ===
N breakthroughs with valid comparisons: 21

β₁ z-scores: mean=-1.772, t=-2.462, p=0.0230
PE z-scores: mean=-26.020, t=-4.840, p=0.0001

Wilcoxon β₁: W=47.0, p=0.0158
Wilcoxon PE: W=28.0, p=0.0014

=== Holm-Bonferroni Correction (4 tests) ===
  PE t-test: p=0.0001, threshold=0.0125 → **SIGNIFICANT**
  PE Wilcoxon: p=0.0014, threshold=0.0167 → **SIGNIFICANT**
  β₁ Wilcoxon: p=0.0158, threshold=0.0250 → **SIGNIFICANT**
  β₁ t-test: p=0.0230, threshold=0.0500 → **SIGNIFICANT**

Cohen's d (β₁, raw): -5.420
Cohen's d (PE, raw):  -10.480

=== Caveats ===
- Observations are non-independent: breakthroughs share topology pairs
- Mean matching pairs per breakthrough: 9.9
- Effective sample size is smaller than N=21
- 8/34 breakthroughs skipped (non-priority CPC pairs)

** At least one test survives Holm-Bonferroni correction **


## Figure 4: ROC-Like Curve

If we treat high β₁ as a "breakthrough detector", how well does it discriminate?

In [10]:
# %% Figure 4: Detection performance
from sklearn.metrics import roc_curve, roc_auc_score

if len(comp_df) > 0 and len(null_baseline) > 0:
    # Build a proper binary classification: label real pre-breakthrough as 1, null as 0
    real_values = comp_df['pre_beta1'].values
    null_values = null_baseline['beta_1'].values
    
    all_values = np.concatenate([real_values, null_values])
    all_labels = np.concatenate([np.ones(len(real_values)), np.zeros(len(null_values))])
    
    # Use sklearn's roc_curve for correct computation
    fpr, tpr, thresholds = roc_curve(all_labels, all_values)
    auc = roc_auc_score(all_labels, all_values)
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(fpr, tpr, color=PALETTE['red'], linewidth=2, label=f'β₁ detector (AUC = {auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC = 0.5)')
    
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Figure 4: β₁ as Breakthrough Detector')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    
    # Note the caveat
    ax.text(0.55, 0.15, f'N real = {len(real_values)}, N null = {len(null_values)}',
            fontsize=9, transform=ax.transAxes, alpha=0.6)
    
    fig.tight_layout()
    save_figure(fig, '04_roc_curve')

2026-03-20 20:50:05 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_roc_curve.png


---
## §5 Robustness Checks — Confound Analysis

The main finding (topology systematically lower before breakthroughs, p < 0.025 for all four
Holm-Bonferroni-corrected tests) warrants robustness verification against the highest-priority
confounds identified in `CONFOUNDS.md`.

| # | Confound | Status |
|---|----------|--------|
| 1 | Examiner citations (~74% of edges may not reflect inventor knowledge) | §5.1 |
| 8 | Assignee self-citations (corporate portfolio structure mimics cross-field flow) | §5.2 |
| 2 | Prosecution lag (grant dates lag filing dates by ~2-3 yr) | §5.3 |
| 3 | Policy shocks (Alice 2014, AIA 2011 may drive topological discontinuities) | §5.4 |

**Prerequisite:** Run `python 00b_build_filtered_citations.py` once before executing this
section. That script builds the filtered datasets from the raw PatentsView TSVs (~30-60 min).

Each subsection ends with a **Verdict** cell: ✓ signal survives / ~ weakens / ✗ disappears.

In [ ]:
# %% §5.6 Confound #9 — Citation Truncation (scope note + epoch truncation check)
# Recent windows (2019-2023) have had less time to accumulate citations.
# This underestimates connectivity → suppresses β₁ in the rightmost epoch windows.
#
# SCOPE: The main precursor test (§3) uses precursor windows ending at filing_year ≤ 2015.
# The earliest window affected by truncation is approximately 2019. So §3 statistics
# are NOT affected. The epoch VISUALIZATION's right tail (+4 to +5 years for ~2015
# breakthroughs) may be artificially low.
#
# Fix: re-plot the superposed epoch using only topology windows with window_end ≤ 2018.

print('=== §5.6: Citation Truncation — Epoch Visualization Check ===')
print('Main precursor statistics (§3) are unaffected: all precursor windows end ≤ 2015.')
print('Checking impact on the epoch visualization right tail...\n')

# Build truncated topology results (cap at 2018)
_topo_truncated = {
    k: v[v['window_end'] <= 2018].copy() if 'window_end' in v.columns else v
    for k, v in topology_results.items()
}

_epoch_full = superposed_epoch(breakthroughs, topology_results, 'beta_1', 10, 5)
_epoch_trunc = superposed_epoch(breakthroughs, _topo_truncated, 'beta_1', 10, 5)

fig56, ax56 = plt.subplots(figsize=(12, 6))

if len(_epoch_full) > 0:
    ax56.plot(_epoch_full['epoch_year'], _epoch_full['mean'],
              color=PALETTE['blue'], linewidth=2, label='Full (through 2023)', alpha=0.8)
if len(_epoch_trunc) > 0:
    ax56.plot(_epoch_trunc['epoch_year'], _epoch_trunc['mean'],
              color=PALETTE['red'], linewidth=2, linestyle='--',
              label='Truncated at 2018', alpha=0.8)

ax56.axvline(0, color='gray', linestyle=':', alpha=0.7, label='Breakthrough (t=0)')
ax56.axvline(3, color='orange', linestyle=':', alpha=0.5, label='Truncation starts (~2019 for 2015 bt)')
ax56.set_xlabel('Years Relative to Breakthrough Filing')
ax56.set_ylabel('β₁ (Mean)')
ax56.set_title('§5.6: Epoch with Full vs. Truncated Topology (2018 cutoff)')
ax56.legend()
fig56.tight_layout()
save_figure(fig56, '04_s5_truncation_check')

print('If the two curves overlap through t=+3 but diverge at t=+4/+5, truncation')
print('only affects the post-breakthrough tail — the pre-breakthrough signal is clean.')
print('\nNote for README: Post-breakthrough topology (t > +3) may underestimate connectivity')
print('due to citation accumulation lag in windows 2019-2023. This does not affect')
print('the pre-breakthrough precursor test statistics.')

In [ ]:
# %% §5.5 Confound #5 — Citation Culture Drift (using mean_distance proxy)
# mean_distance is already in every topology cache window. No new data needed.
from scipy.stats import spearmanr as _sp55

print('=== §5.5: Citation Culture Drift — mean_distance vs β₁ ===')
print('(mean_distance is the inverse of citation density in the co-citation matrix)\n')

_dist_corr_results = {}
for _pk, _topo in topology_results.items():
    if 'x' not in _pk:
        continue
    if 'mean_distance' not in _topo.columns or 'beta_1' not in _topo.columns:
        continue
    _valid55 = _topo[['mean_distance', 'beta_1']].dropna()
    if len(_valid55) < 5:
        continue
    _rho55, _p55 = _sp55(_valid55['mean_distance'], _valid55['beta_1'])
    _dist_corr_results[_pk] = (_rho55, _p55)

# Print and flag significant correlations
_n_sig55 = sum(1 for r, p in _dist_corr_results.values() if p < 0.05)
print(f'Pairs with significant mean_distance ~ β₁ correlation: {_n_sig55}/{len(_dist_corr_results)}')
print()
print(f'{"Pair":<8} {"ρ":>7} {"p":>8}  {"sig":>5}')
print('-' * 35)
for _pk, (_rho55, _p55) in sorted(_dist_corr_results.items()):
    _sig55 = '  *' if _p55 < 0.05 else ''
    print(f'{_pk:<8} {_rho55:>7.3f} {_p55:>8.4f}{_sig55}')

# Per-breakthrough: does mean_distance dip in precursor window alongside β₁?
print('\n=== Pre-breakthrough mean_distance vs β₁ z-scores ===')
_dist_vals = []
for _, _row in comp_df.iterrows():
    _bt = next((b for b in breakthroughs if b.name == _row['name']), None)
    if _bt is None:
        _dist_vals.append(np.nan)
        continue
    _start55, _end55 = get_precursor_window(_bt, years_before=10)
    _dv = []
    for _pk, _topo in topology_results.items():
        if 'x' not in _pk:
            continue
        _sa55, _sb55 = _pk.split('x')
        if not any(s in [_sa55, _sb55] for s in _bt.cpc_sections):
            continue
        if 'window_end' not in _topo.columns or 'mean_distance' not in _topo.columns:
            continue
        _pre55 = _topo[(_topo['window_end'] >= _start55) & (_topo['window_end'] <= _end55)]
        if len(_pre55) > 0:
            _dv.append(_pre55['mean_distance'].mean())
    _dist_vals.append(float(np.nanmean(_dv)) if _dv else np.nan)

_dist_series = pd.Series(_dist_vals, index=comp_df.index)

# Z-score mean_distance vs null
_null_dist_vals = []
for _, _row in comp_df.iterrows():
    _null = null_results.get(_row['name'])
    if _null is None or 'mean_distance' not in _null.columns:
        _null_dist_vals.append(np.nan)
        continue
    _null_dist_vals.append(_null['mean_distance'].mean())
_null_dist_series = pd.Series(_null_dist_vals, index=comp_df.index)

_pre_dist_arr = _dist_series.dropna()
_null_dist_arr = _null_dist_series[_pre_dist_arr.index].dropna()
_idx_both = _pre_dist_arr.index.intersection(_null_dist_arr.index)

if len(_idx_both) >= 5:
    _t55, _p55t = stats.ttest_1samp(
        _pre_dist_arr[_idx_both] - _null_dist_arr[_idx_both], 0
    )
    print(f'Pre-breakthrough mean_distance vs null (N={len(_idx_both)}):')
    print(f'  Mean difference (pre - null): {(_pre_dist_arr[_idx_both] - _null_dist_arr[_idx_both]).mean():.4f}')
    print(f'  t-test: t={_t55:.3f}, p={_p55t:.4f}')

# Correlation between β₁ z-score and mean_distance z-score (across breakthroughs)
_common_idx = comp_df.index[_dist_series.notna()]
if len(_common_idx) >= 5:
    _rho_cross, _p_cross = _sp55(
        comp_df.loc[_common_idx, 'z_beta1'],
        _dist_series[_common_idx]
    )
    print(f'\nCross-breakthrough correlation (z_beta1 ~ pre-breakthrough mean_distance):')
    print(f'  ρ={_rho_cross:.3f}, p={_p_cross:.4f}')
    if abs(_rho_cross) > 0.5 and _p_cross < 0.05:
        print('  ⚠ Strong correlation — topology signal may be driven by citation density.')
    else:
        print('  ✓ Weak correlation — β₁ captures structure beyond citation density.')

print('\n=== §5.5 Verdict ===')
print(f'  {_n_sig55}/{len(_dist_corr_results)} pairs: mean_distance significantly predicts β₁ over time')
print('  If few pairs show correlation, and z_beta1 does not correlate with mean_distance,')
print('  the topological signal is not simply a density artifact.')

### §5.5 Confound #5 — Citation Culture Drift

Different CPC sections have different citation norms (pharma cites broadly; mechanical cites narrowly). These norms also drift over time as examination guidelines evolve. If β₁ is just tracking citation density rather than topological structure, it would correlate tightly with `mean_distance` (lower distance = denser co-citation = more potential loops).

The scale-normalization from Notebook 02 already removes the bulk density effect, but residual citation culture drift within pairs could remain.

**Test:** Spearman correlation between `mean_distance` and `beta_1` across windows for each CPC pair. Also, for each breakthrough, check whether the pre-breakthrough dip in β₁ is accompanied by a dip in `mean_distance` — if both dip together, density is confounded with topology.

In [ ]:
# %% §5.0 Load filtered citation datasets
# Requires: python 00b_build_filtered_citations.py (run once, ~30-60 min)
from src.confounds import examiner_fraction_by_window, partial_out_confound

def _load_filtered_dataset(name: str):
    """Load a filtered citation parquet; return None with a warning if missing."""
    _path = DATA_DIR / name
    if not _path.exists():
        print(f'  MISSING: {name}')
        return None
    _df = pd.read_parquet(_path)
    if 'citing_date' in _df.columns and 'citing_year' not in _df.columns:
        _df['citing_year'] = pd.to_datetime(_df['citing_date']).dt.year
    print(f'  {name}: {len(_df):,} rows')
    return _df

print('Loading filtered citation datasets...')
_citations_with_category = _load_filtered_dataset('citations_with_category.parquet')
_citations_applicant_only = _load_filtered_dataset('citations_applicant_only.parquet')
_patent_assignee = _load_filtered_dataset('patent_assignee.parquet')
_citations_no_self_cite = _load_filtered_dataset('citations_no_self_cite.parquet')
_patent_filing_dates = _load_filtered_dataset('patent_filing_dates.parquet')

_any_missing = any(x is None for x in [
    _citations_with_category, _citations_applicant_only,
    _patent_assignee, _citations_no_self_cite, _patent_filing_dates
])
if _any_missing:
    print('\nOne or more datasets missing. Run: python 00b_build_filtered_citations.py')
else:
    print('\nAll filtered datasets loaded.')

# Report null model cache status
from src.nullmodel import NULL_CACHE as _NULL_CACHE
_null_pkl_count = len(list(_NULL_CACHE.glob('*.pkl')))
print(f'\nNull model cache: {_null_pkl_count} pickle files')
if _null_pkl_count == 0:
    print('  Null models were computed fresh during this notebook run.')
    print('  NOTE: This run reflects the corrected null model (single-section')
    print('  breakthroughs now matched against cross-section pairs, not global topology).')

### §5.1 Confound #1 — Examiner Citations

~40-74% of patent citations are added by USPTO examiners, not inventors. Our data shows the
raw split is `"cited by examiner"` vs. `"cited by applicant"` in `g_us_patent_citation.tsv`.
If topological signals correlate with examiner behavior rather than inventor knowledge, the
finding is an artifact of examination practices, not discovery.

**Test strategy:**
- §5.1a: Compute examiner fraction per CPC pair × sliding window; test Spearman correlation
  with β₁ (does examiner fraction predict topology shape?)
- §5.1b: Partial out the linear effect of examiner fraction from each breakthrough's z-score
  using OLS residuals; re-run the one-sample t-test and Wilcoxon on residuals
- §5.1c: Verdict — does significance survive after controlling for examiner patterns?

In [ ]:
# %% §5.1 Confound #1 — Examiner Citation Analysis
from scipy.stats import spearmanr
from src.confounds import examiner_fraction_by_window, partial_out_confound

_FOCUS_PAIRS = [('A', 'C'), ('A', 'G'), ('A', 'H'), ('C', 'G'), ('C', 'H'), ('G', 'H')]

if _citations_with_category is None:
    print('Skipping §5.1: citations_with_category.parquet not found. Run 00b_build_filtered_citations.py')
else:
    # 5.1a: Examiner fraction time series for focus pairs
    print('=== §5.1a: Examiner Citation Fraction by CPC Pair ===')
    _examiner_fracs = {}
    for _sa, _sb in _FOCUS_PAIRS:
        _ef = examiner_fraction_by_window(_citations_with_category, cpc_map, _sa, _sb)
        _pk = f'{_sa}x{_sb}'
        _examiner_fracs[_pk] = _ef
        _valid_ef = _ef.dropna(subset=['examiner_fraction'])
        if len(_valid_ef) > 0:
            print(f'  {_pk}: mean examiner fraction = {_valid_ef["examiner_fraction"].mean():.3f}')

    # Spearman correlation: examiner_fraction ~ beta_1
    print('\nSpearman correlation (examiner_fraction vs β₁):')
    _spearman_results = {}
    for _pk, _ef in _examiner_fracs.items():
        _topo = topology_results.get(_pk)
        if _topo is None or len(_topo) == 0:
            continue
        if 'window_end' not in _ef.columns or 'window_end' not in _topo.columns:
            continue
        _merged_ef = _ef.merge(_topo[['window_end', 'beta_1']], on='window_end', how='inner')
        _merged_ef = _merged_ef.dropna(subset=['examiner_fraction', 'beta_1'])
        if len(_merged_ef) < 5:
            continue
        _rho, _p_rho = spearmanr(_merged_ef['examiner_fraction'], _merged_ef['beta_1'])
        _spearman_results[_pk] = (_rho, _p_rho)
        _sig = ' *' if _p_rho < 0.05 else ''
        print(f'  {_pk}: ρ={_rho:.3f}, p={_p_rho:.4f}{_sig}')

    # 5.1b: Partial-out examiner fraction from z_beta1
    print('\n=== §5.1b: Residual Test (Examiner Fraction Partialed Out) ===')
    _bt_ef = []
    for _, _row in comp_df.iterrows():
        _bt = next((b for b in breakthroughs if b.name == _row['name']), None)
        if _bt is None:
            _bt_ef.append(np.nan)
            continue
        _start, _end = get_precursor_window(_bt, years_before=10)
        _ef_vals = []
        for _sa, _sb in _FOCUS_PAIRS:
            if not any(s in [_sa, _sb] for s in _bt.cpc_sections):
                continue
            _pk = f'{_sa}x{_sb}'
            if _pk not in _examiner_fracs:
                continue
            _pre_ef = _examiner_fracs[_pk]
            _pre_ef = _pre_ef[
                (_pre_ef['window_end'] >= _start) & (_pre_ef['window_end'] <= _end)
            ]['examiner_fraction']
            if len(_pre_ef) > 0:
                _ef_vals.append(_pre_ef.dropna().mean())
        _bt_ef.append(float(np.nanmean(_ef_vals)) if _ef_vals else np.nan)

    _ef_series = pd.Series(_bt_ef, index=comp_df.index, name='examiner_fraction')
    print(f'Breakthroughs with examiner fraction data: {_ef_series.notna().sum()}/{len(comp_df)}')
    print(f'Mean pre-breakthrough examiner fraction: {_ef_series.mean():.3f}')

    _z_resid_b1 = partial_out_confound(comp_df['z_beta1'], _ef_series)
    _valid_resid = _z_resid_b1.dropna()

    _t_resid, _p_resid_t = stats.ttest_1samp(_valid_resid, 0)
    try:
        _w_resid, _p_resid_w = stats.wilcoxon(_valid_resid)
    except ValueError:
        _p_resid_w = 1.0

    print(f'\nResidual β₁ z-scores (examiner fraction effect removed):')
    print(f'  mean residual z = {_valid_resid.mean():.3f}')
    print(f'  t-test: t={_t_resid:.3f}, p={_p_resid_t:.4f}')
    print(f'  Wilcoxon: W={_w_resid:.1f}, p={_p_resid_w:.4f}')

    # 5.1c: Verdict
    print('\n=== §5.1c Verdict ===')
    _n_sig_sp = sum(1 for _r, _p in _spearman_results.values() if _p < 0.05)
    print(f'  {_n_sig_sp}/{len(_spearman_results)} pairs: examiner fraction significantly correlated with β₁')
    if _p_resid_t < 0.05 and _p_resid_w < 0.05:
        print('  ✓ Signal SURVIVES after partialing out examiner fraction.')
        print('    Pre-breakthrough topology decrease is not explained by examiner citation patterns.')
    elif _p_resid_t < 0.1 or _p_resid_w < 0.1:
        print('  ~ Signal WEAKENS but partially persists after partialing out examiner fraction.')
    else:
        print('  ✗ Signal does NOT survive after partialing out examiner fraction.')
        print('    The topology decrease may reflect examination patterns, not inventor knowledge flow.')

### §5.2 Confound #8 — Assignee Self-Citations

Large companies (IBM, Samsung, Qualcomm, Merck) cite their own prior patents heavily. When
two divisions of the same company operate in different CPC sections, their cross-section
citations look like interdisciplinary knowledge flow but may reflect portfolio management.

**Test strategy:**
- §5.2a: Measure intra-assignee fraction per CPC pair × window; test correlation with β₁
- §5.2b: Re-run TDA for 4 highest-impact pairs using `citations_no_self_cite.parquet`;
  compare β₁ time series and re-run precursor test
- §5.2c: Verdict

> **Note on §5.2b:** Re-running TDA for 4 pairs takes ~2-4 hours on a 16 GB Mac. Results
> are cached to `data/topology_cache_no_self_cite/`. If the cache exists from a prior run,
> this cell loads instantly.

In [ ]:
# %% §5.2a Confound #8 — Self-Citation Fraction Characterization
from scipy.stats import spearmanr as _spearmanr

if _citations_no_self_cite is None or _patent_assignee is None:
    print('Skipping §5.2a: required files not found')
else:
    from src.confounds import self_citation_fraction_by_window

    # Reload base citations with citing_year for self-cite computation
    _base_cits_sc = pd.read_parquet(DATA_DIR / 'citations.parquet')
    if 'citing_year' not in _base_cits_sc.columns:
        _base_cits_sc['citing_year'] = pd.to_datetime(_base_cits_sc['citing_date']).dt.year

    print('=== §5.2a: Intra-Assignee Citation Fraction by CPC Pair ===')
    _self_cite_fracs = {}
    for _sa, _sb in _FOCUS_PAIRS:
        _scf = self_citation_fraction_by_window(
            _base_cits_sc, _patent_assignee, cpc_map, _sa, _sb
        )
        _pk = f'{_sa}x{_sb}'
        _self_cite_fracs[_pk] = _scf
        _valid_scf = _scf.dropna(subset=['self_cite_fraction'])
        if len(_valid_scf) > 0:
            print(f'  {_pk}: mean self-cite fraction = {_valid_scf["self_cite_fraction"].mean():.3f}')

    print('\nSpearman correlation (self_cite_fraction vs β₁):')
    for _pk, _scf in _self_cite_fracs.items():
        _topo = topology_results.get(_pk)
        if _topo is None or len(_topo) == 0:
            continue
        _merged_sc = _scf.merge(_topo[['window_end', 'beta_1']], on='window_end', how='inner')
        _merged_sc = _merged_sc.dropna(subset=['self_cite_fraction', 'beta_1'])
        if len(_merged_sc) < 5:
            continue
        _rho_sc, _p_sc = _spearmanr(_merged_sc['self_cite_fraction'], _merged_sc['beta_1'])
        _sig_sc = ' *' if _p_sc < 0.05 else ''
        print(f'  {_pk}: ρ={_rho_sc:.3f}, p={_p_sc:.4f}{_sig_sc}')

    print('\nNote: §5.2b below re-runs TDA on citations_no_self_cite for the 4 key pairs (slow).')

In [ ]:
# %% §5.2b Confound #8 — Targeted Topology Re-run (No Intra-Assignee Citations)
# WARNING: If topology is NOT cached, this cell takes ~2-4 hours on a 16 GB Mac.
# Results are cached to data/topology_cache_no_self_cite/ — safe to re-run.
if _citations_no_self_cite is None:
    print('Skipping §5.2b: citations_no_self_cite.parquet not found. Run 00b_build_filtered_citations.py')
else:
    _TARGET_PAIRS = [('A', 'C'), ('A', 'G'), ('C', 'H'), ('G', 'H')]
    _CACHE_NO_SELF = str(DATA_DIR / 'topology_cache_no_self_cite')

    print('=== §5.2b: Topology Re-run (Intra-Assignee Citations Removed) ===')
    print(f'Target pairs: {[f"{a}x{b}" for a, b in _TARGET_PAIRS]}')
    print(f'Cache dir: {_CACHE_NO_SELF}')
    print('Loading from cache if available; computing fresh otherwise...')

    _pair_results_no_self = compute_all_priority_pairs(
        _citations_no_self_cite, cpc_map,
        cache_dir=_CACHE_NO_SELF,
        pairs=_TARGET_PAIRS,
        start_year=1980,
        end_year=2023,
    )

    print(f'\nComputed {len(_pair_results_no_self)} window-pair observations')

    # Overlay comparison: original vs. no-self-cite β₁
    fig52b, axes52b = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
    axes52b = axes52b.flatten()

    for i, (sa, sb) in enumerate(_TARGET_PAIRS):
        ax = axes52b[i]
        pk = f'{sa}x{sb}'
        _orig = topology_results.get(pk)
        _nsc = _pair_results_no_self[
            _pair_results_no_self['section_pair'] == pk
        ] if 'section_pair' in _pair_results_no_self.columns else pd.DataFrame()

        if _orig is not None and 'window_end' in _orig.columns:
            ax.plot(_orig['window_end'], _orig['beta_1'],
                    color=PALETTE['blue'], linewidth=2, label='Original', alpha=0.8)
        if len(_nsc) > 0 and 'window_end' in _nsc.columns:
            ax.plot(_nsc['window_end'], _nsc['beta_1'],
                    color=PALETTE['red'], linewidth=2, linestyle='--',
                    label='No intra-assignee', alpha=0.8)

        ax.set_title(pk)
        ax.set_ylabel('β₁')
        ax.set_xlabel('Year')
        if i == 0:
            ax.legend(fontsize=9)

    fig52b.suptitle('§5.2b: β₁ Original vs. Self-Cite-Filtered (4 Key CPC Pairs)', fontsize=13)
    fig52b.tight_layout()
    save_figure(fig52b, '04_s5_no_self_cite')

    # Re-run precursor test on no-self-cite topology vs. existing null models
    _no_self_topo_results = {}
    for _pair, _group in _pair_results_no_self.groupby('section_pair'):
        _no_self_topo_results[_pair] = _group.reset_index(drop=True)
        _no_self_topo_results[_pair.replace('x', '_')] = _group.reset_index(drop=True)

    _no_self_comp = []
    for _, _row in comp_df.iterrows():
        _bt = next((b for b in breakthroughs if b.name == _row['name']), None)
        if _bt is None:
            continue
        _secs = _bt.cpc_sections
        _start, _end = get_precursor_window(_bt, years_before=10)
        _pre_vals = []
        for _k, _t in _no_self_topo_results.items():
            if 'x' not in _k:
                continue
            _sa2, _sb2 = _k.split('x')
            if not any(s in [_sa2, _sb2] for s in _secs):
                continue
            if 'window_end' not in _t.columns:
                continue
            _pre = _t[(_t['window_end'] >= _start) & (_t['window_end'] <= _end)]
            if len(_pre) > 0 and 'beta_1' in _pre.columns:
                _pre_vals.append(_pre['beta_1'].mean())
        if not _pre_vals:
            continue
        _null = null_results.get(_bt.name)
        if _null is None:
            continue
        _pb1 = np.mean(_pre_vals)
        _nm = _null['beta_1'].mean()
        _ns = _null['beta_1'].std()
        _z = (_pb1 - _nm) / _ns if _ns > 1e-6 else 0.0
        _no_self_comp.append({'name': _bt.name, 'z_beta1_no_self': _z})

    _no_self_comp_df = pd.DataFrame(_no_self_comp)
    if len(_no_self_comp_df) > 0:
        _t_ns, _p_ns_t = stats.ttest_1samp(_no_self_comp_df['z_beta1_no_self'], 0)
        try:
            _w_ns, _p_ns_w = stats.wilcoxon(_no_self_comp_df['z_beta1_no_self'])
        except ValueError:
            _p_ns_w = 1.0
        print(f'\nPrecursor test (no intra-assignee, N={len(_no_self_comp_df)}):')
        print(f'  β₁ t-test: p={_p_ns_t:.4f}')
        print(f'  β₁ Wilcoxon: p={_p_ns_w:.4f}')
        print('\n=== §5.2c Verdict ===')
        if _p_ns_t < 0.05 or _p_ns_w < 0.05:
            print('  ✓ Signal SURVIVES after removing intra-assignee citations.')
            print('    The topology decrease before breakthroughs is not explained by self-citation clustering.')
        else:
            print('  ~ Signal weakens or disappears after removing intra-assignee citations.')
            print('    Cross-section topology may partly reflect corporate portfolio structure.')
    else:
        print('\n  Could not re-run test (no overlap between no-self-cite pairs and breakthrough catalog)')

### §5.3 Confound #2 — Prosecution Lag (Filing Date Substitution)

Grant dates lag filing dates by ~2-3 years on average (longer for biotech, shorter for software).
Our topology is indexed by grant year. If the "pre-breakthrough" topological signal is real, it
should appear BEFORE the filing year — not just before the grant year.

**Test:** Re-run the superposed epoch plot at three alignment points — filing year (original),
filing year + 2yr (approximates grant year), and filing year − 2yr (pre-filing) — and compare
whether the dip is present at all three. This does **not** require recomputing TDA — only the
epoch alignment changes.

In [ ]:
# %% §5.3 Confound #2 — Prosecution Lag Sensitivity
if _patent_filing_dates is None:
    print('Skipping §5.3: patent_filing_dates.parquet not found. Run 00b_build_filtered_citations.py')
else:
    print('=== §5.3: Prosecution Lag by CPC Section ===')

    _patents_lag = pd.read_parquet(DATA_DIR / 'patents.parquet')[['patent_id', 'date']]
    _lag_df = _patents_lag.merge(_patent_filing_dates, on='patent_id', how='inner')
    _lag_df['lag_years'] = (
        pd.to_datetime(_lag_df['date']) - pd.to_datetime(_lag_df['filing_date'])
    ).dt.days / 365.25
    _lag_df = _lag_df[(_lag_df['lag_years'] >= 0) & (_lag_df['lag_years'] <= 15)]

    _cpc_primary_map = cpc_map.drop_duplicates('patent_id').set_index('patent_id')['cpc_section']
    _lag_df['cpc_section'] = _lag_df['patent_id'].map(_cpc_primary_map)
    _lag_df = _lag_df.dropna(subset=['cpc_section'])

    _lag_by_section = _lag_df.groupby('cpc_section')['lag_years'].agg(
        median_lag='median', mean_lag='mean', n_patents='count'
    )
    print(_lag_by_section.to_string())

    # Three-panel superposed epoch: filing_year, grant_year approx, pre-filing
    _adjustments = {
        'Align at filing year\n(original)': 0,
        'Align at grant year\n(filing + 2 yr)': 2,
        'Align pre-filing\n(filing − 2 yr)': -2,
    }

    fig53, axes53 = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    for ax, (label, adj) in zip(axes53, _adjustments.items()):
        class _AdjBT:
            def __init__(self, bt, a):
                self.name = bt.name
                self.category = bt.category
                self.cpc_sections = bt.cpc_sections
                self.filing_year = bt.filing_year + a
                self.recognition_year = bt.recognition_year

        _adj_bts = [_AdjBT(bt, adj) for bt in breakthroughs]
        _epoch_adj = superposed_epoch(
            breakthroughs=_adj_bts,
            topology_results=topology_results,
            metric='beta_1',
            years_before=10,
            years_after=5,
        )

        if len(_epoch_adj) > 0:
            ax.plot(_epoch_adj['epoch_year'], _epoch_adj['mean'],
                    color=PALETTE['red'], linewidth=2)
            ax.fill_between(
                _epoch_adj['epoch_year'],
                _epoch_adj['mean'] - _epoch_adj['std'],
                _epoch_adj['mean'] + _epoch_adj['std'],
                alpha=0.2, color=PALETTE['red']
            )

        if len(null_baseline) > 0:
            _nm = null_baseline['beta_1'].mean()
            _ns = null_baseline['beta_1'].std()
            ax.axhspan(_nm - 1.96 * _ns, _nm + 1.96 * _ns, alpha=0.1, color=PALETTE['blue'])
            ax.axhline(_nm, color=PALETTE['blue'], linestyle='--', alpha=0.5)

        ax.axvline(0, color='gray', linestyle=':', alpha=0.7)
        ax.set_xlabel('Years Relative to Alignment Point')
        ax.set_ylabel('β₁')
        ax.set_title(label, fontsize=11)

    fig53.suptitle(
        '§5.3: Prosecution Lag Sensitivity — Superposed Epoch at Three Alignments\n'
        'If the dip shifts/disappears at +2yr alignment, the signal may be post-filing activity.',
        fontsize=12
    )
    fig53.tight_layout()
    save_figure(fig53, '04_s5_prosecution_lag')

    print('\n=== §5.3 Verdict ===')
    print('Interpret the three panels:')
    print('  Panel 1 (filing year): current main result')
    print('  Panel 2 (+2yr, grant approx): if the dip disappears, signal may be prosecution-driven')
    print('  Panel 3 (-2yr, pre-filing): if a dip appears here too, signal predates the filing itself')
    print(f'\n  Median prosecution lag across all sections: {_lag_df["lag_years"].median():.2f} yr')
    print(f'  Longest lag: {_lag_by_section["median_lag"].idxmax()} ({_lag_by_section["median_lag"].max():.2f} yr)')

### §5.4 Confound #3 — Policy Shocks (Alice, AIA)

Discrete legal events reshaped the patent landscape in ways unrelated to knowledge flow:
- **AIA (2011-09-16):** Changed US from first-to-invent to first-inventor-to-file
- **Alice Corp. v. CLS Bank (2014-06-19):** Dramatically narrowed software patent eligibility, sharply reducing grants in CPC sections G and H

These could create topological discontinuities that mimic breakthrough precursor signatures,
or could coincide with breakthrough filing years by chance.

**Test:** (a) Visual inspection of β₁ time series with policy dates marked. (b) t-test of mean β₁
in ±2yr windows around policy events vs. the rest of the series. (c) Exclude breakthroughs whose
filing year falls within ±3 years of a policy shock and re-run the main test.

In [ ]:
# %% §5.4 Confound #3 — Policy Shocks (Alice, AIA)
from src.confounds import policy_shock_dates

_policy_dates = policy_shock_dates()
POLICY_PAIRS_54 = [('G', 'H'), ('B', 'G'), ('A', 'H'), ('C', 'H')]

print('=== §5.4: Topological Discontinuities at Policy Shock Dates ===')
print('Policy events:', _policy_dates)

_alice_year = 2014
_aia_year = 2011

# Breakthroughs within ±3 years of policy shocks
print('\nBreakthroughs within ±3 years of policy shocks:')
_flagged_bts = []
for bt in breakthroughs:
    _near = []
    if abs(bt.filing_year - _alice_year) <= 3:
        _near.append(f'Alice ({_alice_year}, Δ={bt.filing_year - _alice_year:+d}yr)')
    if abs(bt.filing_year - _aia_year) <= 3:
        _near.append(f'AIA ({_aia_year}, Δ={bt.filing_year - _aia_year:+d}yr)')
    if _near:
        _flagged_bts.append(bt.name)
        print(f'  {bt.name} ({bt.filing_year}) — near: {", ".join(_near)}')

if not _flagged_bts:
    print('  None.')

# Plot β₁ for policy-affected pairs with event markers
_nrows54 = (len(POLICY_PAIRS_54) + 1) // 2
fig, axes = plt.subplots(_nrows54, 2, figsize=(14, 5 * _nrows54), sharex=True)
axes = axes.flatten()

for i, (sa, sb) in enumerate(POLICY_PAIRS_54):
    ax = axes[i]
    pair_key = f'{sa}x{sb}'
    topo = topology_results.get(pair_key)

    if topo is not None and 'window_end' in topo.columns:
        ax.plot(topo['window_end'], topo['beta_1'],
                color=PALETTE['blue'], linewidth=2, label='β₁')
        ax.axvline(_aia_year, color='orange', linestyle='--', linewidth=2,
                   alpha=0.8, label='AIA (2011)')
        ax.axvline(_alice_year, color='red', linestyle='--', linewidth=2,
                   alpha=0.8, label='Alice (2014)')
        for bt in breakthroughs:
            if any(s in [sa, sb] for s in bt.cpc_sections):
                ax.axvline(bt.filing_year, color='gray', linestyle=':', alpha=0.3, linewidth=1)

    ax.set_title(f'{pair_key} — β₁ with Policy Shock Dates')
    ax.set_ylabel('β₁')
    ax.set_xlabel('Year')
    if i == 0:
        ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('§5.4: Policy Shocks vs. Topological Structure', fontsize=13)
fig.tight_layout()
save_figure(fig, '04_s5_policy_shocks')

# Statistical test: β₁ near vs. far from policy dates
print('\nTopology level around policy events (mean β₁ in ±2yr window vs. rest):')
for pair_key in [f'{a}x{b}' for a, b in POLICY_PAIRS_54]:
    topo = topology_results.get(pair_key)
    if topo is None or 'window_end' not in topo.columns:
        continue
    print(f'\n  {pair_key}:')
    for event, date_str in _policy_dates.items():
        event_year = int(date_str[:4])
        near = topo[abs(topo['window_end'] - event_year) <= 2]['beta_1']
        far = topo[abs(topo['window_end'] - event_year) > 2]['beta_1']
        if len(near) < 2 or len(far) < 2:
            continue
        t54, p54 = stats.ttest_ind(near, far)
        sig = ' *' if p54 < 0.05 else ''
        print(f'    {event} ({event_year}): near={near.mean():.3f}, far={far.mean():.3f}, p={p54:.4f}{sig}')

# Sensitivity: re-run main test without policy-adjacent breakthroughs
print('\n=== §5.4 Verdict ===')
if _flagged_bts:
    _comp_no_policy = comp_df[~comp_df['name'].isin(_flagged_bts)]
    print(f'Excluding {len(_flagged_bts)} policy-adjacent breakthroughs → N={len(_comp_no_policy)}')
    if len(_comp_no_policy) >= 5:
        _t_np, _p_np = stats.ttest_1samp(_comp_no_policy['z_beta1'], 0)
        try:
            _w_np, _pw_np = stats.wilcoxon(_comp_no_policy['z_beta1'])
        except ValueError:
            _pw_np = 1.0
        print(f'  β₁ t-test (no policy-adjacent): p={_p_np:.4f}')
        print(f'  β₁ Wilcoxon (no policy-adjacent): p={_pw_np:.4f}')
        if _p_np < 0.05 and _pw_np < 0.05:
            print('  ✓ Signal SURVIVES exclusion of policy-adjacent breakthroughs.')
        else:
            print('  ~ Signal weakens when policy-adjacent breakthroughs are excluded.')
            print('    Some of the pre-breakthrough topology decrease may reflect policy shocks.')
    else:
        print('  Insufficient N after exclusion to re-run test.')
else:
    print('  No breakthroughs fall within ±3 years of policy shocks.')
    print('  ✓ Policy shock confound is unlikely to drive the main result.')

## Summary

**The Precursor Test answers the central question:**
Do topological signatures in the patent citation network systematically
precede technological breakthroughs?

The answer depends on the data. If the superposed epoch shows a clear signal
above the null band, topology is a leading indicator. If not, innovation
is topologically invisible until it arrives — which is itself a meaningful
scientific finding.

Either way, this feeds into Notebook 05 where we test whether topology adds
predictive power beyond simpler metrics.

---
## §6 Leave-One-Out Robustness (Outlier Sensitivity)

A finding with N=57 could be driven by a handful of outlier breakthroughs with extreme z-scores.
This section tests whether the result is robust to the removal of any single breakthrough.

**Jackknife approach:** For each breakthrough, remove it from the sample and re-run the
one-sample Wilcoxon signed-rank test on the remaining N-1 z-scores. If the test stays
significant after removing any single observation, the result is not outlier-driven.

We also check: if we restrict to breakthroughs with ≥ 5 precursor windows only, does
significance hold? (Breakthroughs with 1-2 precursor windows have noisy z-scores.)

In [ ]:
# %% §6.1 Jackknife LOO sensitivity
from scipy.stats import wilcoxon as scipy_wilcoxon

if len(comp_df) >= 10:
    z_beta1 = comp_df['z_beta1'].values
    z_pe = comp_df['z_pe'].values
    names = comp_df['name'].values

    loo_results = []
    for i in range(len(z_beta1)):
        z_loo = np.delete(z_beta1, i)
        z_pe_loo = np.delete(z_pe, i)
        try:
            _, p_b1_loo = scipy_wilcoxon(z_loo, alternative='less')
            _, p_pe_loo = scipy_wilcoxon(z_pe_loo, alternative='less')
        except ValueError:
            p_b1_loo, p_pe_loo = float('nan'), float('nan')
        loo_results.append({
            'removed': names[i],
            'mean_z_b1': z_loo.mean(),
            'p_b1': p_b1_loo,
            'p_pe': p_pe_loo,
            'b1_sig': p_b1_loo < 0.05,
            'pe_sig': p_pe_loo < 0.05,
        })

    loo_df = pd.DataFrame(loo_results)
    n_b1_sig = loo_df['b1_sig'].sum()
    n_pe_sig = loo_df['pe_sig'].sum()
    n_total = len(loo_df)

    print('=== JACKKNIFE LEAVE-ONE-OUT ROBUSTNESS ===')
    print(f'N = {len(z_beta1)} breakthroughs')
    print(f'β₁ Wilcoxon p<0.05 in {n_b1_sig}/{n_total} LOO runs ({100*n_b1_sig/n_total:.0f}%)')
    print(f'PE Wilcoxon p<0.05 in {n_pe_sig}/{n_total} LOO runs ({100*n_pe_sig/n_total:.0f}%)')
    print()

    # Find the most influential (if removed, p goes highest)
    most_influential_b1 = loo_df.nlargest(3, 'p_b1')[['removed', 'p_b1', 'mean_z_b1']]
    print('Most influential breakthroughs for β₁ (p rises most when removed):')
    print(most_influential_b1.to_string(index=False))

    if n_b1_sig == n_total:
        print(f'\n✓ β₁ result is robust: removing ANY single breakthrough keeps p<0.05')
    else:
        n_not_sig = n_total - n_b1_sig
        cases = loo_df[~loo_df['b1_sig']][['removed','p_b1']]
        print(f'\n! β₁ result requires all breakthroughs: {n_not_sig} LOO run(s) drop below p=0.05')
        print(cases.to_string(index=False))
else:
    print(f'Too few breakthroughs for jackknife: {len(comp_df)}')

In [ ]:
# %% §6.2 Minimum-window-count sensitivity
if len(comp_df) >= 10 and 'n_precursor_windows' in comp_df.columns:
    print('=== SENSITIVITY BY MINIMUM PRECURSOR WINDOWS ===')
    print(f'Full sample: N={len(comp_df)}')
    for min_w in [3, 5, 7]:
        subset = comp_df[comp_df['n_precursor_windows'] >= min_w]
        if len(subset) < 5:
            print(f'  min_windows>={min_w}: N={len(subset)} — too few')
            continue
        z_sub = subset['z_beta1'].values
        try:
            _, p_w = scipy_wilcoxon(z_sub, alternative='less')
        except ValueError:
            p_w = float('nan')
        _, p_t = __import__('scipy.stats', fromlist=['ttest_1samp']).ttest_1samp(z_sub, 0)
        sig = 'p<0.05' if p_w < 0.05 else 'n.s.'
        print(f'  min_windows>={min_w}: N={len(subset)}, mean_z={z_sub.mean():.3f}, '
              f'Wilcoxon p={p_w:.4f} ({sig}), t-test p={p_t:.4f}')

In [ ]:
# %% §6.3 Category-level breakdown
if len(comp_df) >= 10 and 'category' in comp_df.columns:
    print('=== Z-SCORES BY CATEGORY ===')
    cat_stats = comp_df.groupby('category')['z_beta1'].agg(['mean','std','count']).round(3)
    cat_stats.columns = ['mean_z', 'std_z', 'n']
    print(cat_stats.sort_values('mean_z').to_string())
    print()
    print('Negative mean_z (topology lower than null before breakthrough):')
    neg_cats = cat_stats[cat_stats['mean_z'] < 0].index.tolist()
    pos_cats = cat_stats[cat_stats['mean_z'] >= 0].index.tolist()
    print(f'  Negative: {neg_cats}')
    print(f'  Positive: {pos_cats}')
    if len(pos_cats) == 0:
        print('  ✓ Signal consistent across all technology categories')
    elif len(neg_cats) > len(pos_cats):
        print(f'  Signal present in {len(neg_cats)} categories, absent in {len(pos_cats)}')

In [ ]:
# %% §6.4 Note on Strategy 3 (CPC subclass creation events)
print('=== STRATEGY 3 STATUS: CPC SUBCLASS CREATION EVENTS ===')
print()
# Check how many subclass creation events exist in 1990-2018 window
cpc_with_dates = cpc_map.merge(
    patents[['patent_id','date']].rename(columns={'date':'grant_date'}),
    on='patent_id', how='inner'
)
cpc_with_dates['grant_year'] = pd.to_datetime(cpc_with_dates['grant_date']).dt.year

# First appearance year per subclass
first_year = cpc_with_dates.groupby('cpc_subclass')['grant_year'].min()
n_subclasses = len(first_year)
n_post_1990 = (first_year >= 1990).sum()
print(f'Total CPC subclasses in dataset: {n_subclasses}')
print(f'Subclasses first appearing 1990-2018: {n_post_1990}')
print()
print('The CPC system retroactively classifies historical patents, so most')
print('subclasses appear to have been "created" in 1976 even when the technology')
print('is recent. Only subclasses unique to post-1990 technology (e.g., G16Y for')
print('IoT/ICT applications) show true creation events in our data.')
print()
print('Strategy 3 requires CPC subgroup-level data (not preserved in our cpc_map.parquet).')
print('This is documented as a future direction. The 65-entry curated catalog and')
print('N=', len(comp_df), 'valid comparisons already exceed the expand_sample_size.md target.')

### §6 Verdict

**Leave-one-out jackknife:** Tests whether any single breakthrough drives the result.
If the Wilcoxon p<0.05 in ≥95% of LOO runs, the finding is robust to outliers.

**Minimum window sensitivity:** Breakthroughs with very few precursor windows have noisier
z-scores. If the result holds with min_windows≥5, it's not driven by data-sparse observations.

**Strategy 3 status:** CPC subclass creation events are infeasible with 4-character subclass codes
due to retroactive classification (677 subclasses, most 'created' in 1976). A proper Strategy 3
requires subgroup-level CPC data (~200,000 codes) not preserved in our pipeline. Documented as
a future direction.